In [3]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from fundus_data_toolkit.datasets.generic import get_image_dataset
from multistyleseg.experiments.fundus.ensemble import get_ensemble_model
from multistyleseg.data.fundus.consts import Lesions, ALL_CLASSES
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import cv2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
root_img = Path("/home/clement/Documents/data/IVisionHMR/output/fundus")
dataset = get_image_dataset(root_img, (1536, 1536), precise_autocrop=False)
ensemble = get_ensemble_model()
dataset.return_indices = True

Argument(s) 'always_apply' are not valid for transform MaxSizeTransform
Argument(s) 'always_apply' are not valid for transform PadIfNeeded
Argument(s) 'always_apply' are not valid for transform Normalize


In [5]:
dataloader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=8)

In [6]:
ensemble = get_ensemble_model(model_choices=["UNET", "SEGFORMER", "SERESNET UNET"])
ensemble.eval()
ensemble = ensemble.cuda()

In [7]:
from skimage.measure import label, regionprops, find_contours
import pandas as pd
import numpy as np

outputs_folders = Path("ensemble_inference")
output_segmentations = outputs_folders / "segmentations"
output_lesions_folders = outputs_folders / "lesions"
outputs_folders.mkdir(exist_ok=True, parents=True)
output_lesions_folders.mkdir(exist_ok=True, parents=True)
output_segmentations.mkdir(exist_ok=True, parents=True)


for batch in tqdm(dataloader, total=len(dataloader)):
    indices = batch["index"]
    batch_done = False
    for index in indices:
        # Check if output already exists, skip if so
        filepath = Path(dataset.img_filepath["image"][index])
        relative_path = filepath.relative_to(root_img)
        imageId, laterality = relative_path.parts[0], relative_path.parts[1]
        output_lesion = output_segmentations / f"{imageId}_{laterality}.png"
        if output_lesion.exists():
            batch_done = True
            break
    if batch_done:
        continue
    images = batch["image"].cuda()

    with torch.inference_mode():
        with torch.no_grad():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = ensemble(images)
                predictions = torch.argmax(outputs, dim=1).cpu().numpy()
    for i, index in enumerate(indices):
        filepath = Path(dataset.img_filepath["image"][index])
        relative_path = filepath.relative_to(root_img)
        imageId, laterality = relative_path.parts[0], relative_path.parts[1]
        output_lesion = output_segmentations / f"{imageId}_{laterality}.png"
        output_lesion.parent.mkdir(exist_ok=True, parents=True)
        prediction = predictions[i]
        cv2.imwrite(str(output_lesion), prediction.astype(np.uint8))

  0%|          | 0/459 [00:00<?, ?it/s]

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)


In [8]:
from multiprocessing import Pool
from functools import partial


def extract_lesions(segmentation_map_filepath):
    prediction = cv2.imread(str(segmentation_map_filepath), cv2.IMREAD_UNCHANGED)
    image_recognition = segmentation_map_filepath.stem.split("_")
    imageId, laterality = image_recognition[0], image_recognition[1]
    components = label(prediction)
    props = regionprops(components, intensity_image=prediction)
    lesions_id = [ALL_CLASSES[int(prop.max_intensity) - 1].name for prop in props]
    lesions_contours = []
    for prop in props:
        lesion_mask = (components == prop.label).astype(np.uint8)
        contours = find_contours(lesion_mask, 0.5)
        lesions_contours.append(contours)
    data = dict(
        image_id=[imageId] * len(props),
        laterality=[laterality] * len(props),
        lesion_id=lesions_id,
        image_path=[str(segmentation_map_filepath)] * len(props),
        area=[prop.area for prop in props],
        centroid=[prop.centroid for prop in props],
        bbox=[prop.bbox for prop in props],
        coordinates=[prop.coords for prop in props],
        contours=lesions_contours,
    )
    df = pd.DataFrame(data)
    output_pkl = (output_lesions_folders / segmentation_map_filepath.stem).with_suffix(
        ".pkl"
    )
    df.to_pickle(output_pkl)


segmentation_files = list(output_segmentations.glob("*.png"))

with Pool() as pool:
    list(
        tqdm(
            pool.imap(extract_lesions, segmentation_files),
            total=len(segmentation_files),
        )
    )

  0%|          | 0/3670 [00:00<?, ?it/s]